In [ ]:
"""
=============================================================
FILE 32 — TREE OF THOUGHTS (ToT)
=============================================================

CONCEPTS TAUGHT
----------------
1. Tree of Thoughts
2. Branching Reasoning
3. Multi-Path Exploration
4. Reasoning Search
5. Thought Evaluation
6. Alternative Solutions
7. AI Deliberation
8. Complex Problem Solving
9. Search-Based Reasoning
10. Structured Thinking

CORE IDEA
-----------
Instead of one reasoning path,
the AI explores MULTIPLE possible thoughts.

FLOW
-----
Problem
   ↓
Thought Path A
Thought Path B
Thought Path C
   ↓
Evaluate All Paths
   ↓
Choose Best Reasoning

REAL WORLD USE CASES
---------------------
- Strategic planning
- Math solving
- Coding agents
- Research systems
- Decision intelligence
"""

# ============================================================
# STEP 1 — IMPORTS
# ============================================================

import os

from dotenv import load_dotenv

from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END

from IPython.display import Image, display

# ============================================================
# STEP 2 — LOAD ENV VARIABLES
# ============================================================

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# ============================================================
# STEP 3 — INITIALIZE LLM
# ============================================================

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

# ============================================================
# STEP 4 — DEFINE STATE
# ============================================================

class State(TypedDict):
    problem: str
    thought_path_1: str
    thought_path_2: str
    thought_path_3: str
    final_decision: str

# ============================================================
# STEP 5 — THOUGHT PATH 1
# ============================================================

def thought_path_1(state: State):

    print("\nGenerating Thought Path 1...\n")

    response = llm.invoke(
        f"""
        Solve the following problem using
        analytical reasoning.

        Problem:
        {state['problem']}
        """
    )

    return {
        "thought_path_1": response.content
    }

# ============================================================
# STEP 6 — THOUGHT PATH 2
# ============================================================

def thought_path_2(state: State):

    print("\nGenerating Thought Path 2...\n")

    response = llm.invoke(
        f"""
        Solve the following problem using
        creative reasoning.

        Problem:
        {state['problem']}
        """
    )

    return {
        "thought_path_2": response.content
    }

# ============================================================
# STEP 7 — THOUGHT PATH 3
# ============================================================

def thought_path_3(state: State):

    print("\nGenerating Thought Path 3...\n")

    response = llm.invoke(
        f"""
        Solve the following problem using
        risk-aware reasoning.

        Problem:
        {state['problem']}
        """
    )

    return {
        "thought_path_3": response.content
    }

# ============================================================
# STEP 8 — EVALUATOR NODE
# ============================================================

def evaluator(state: State):

    print("\nEvaluating all reasoning paths...\n")

    response = llm.invoke(
        f"""
        Compare all reasoning paths and select
        the BEST solution.

        PATH 1:
        {state['thought_path_1']}

        PATH 2:
        {state['thought_path_2']}

        PATH 3:
        {state['thought_path_3']}

        Provide:
        - best reasoning path
        - why it is best
        - final recommendation
        """
    )

    return {
        "final_decision": response.content
    }

# ============================================================
# STEP 9 — BUILD GRAPH
# ============================================================

builder = StateGraph(State)

builder.add_node("thought_path_1", thought_path_1)
builder.add_node("thought_path_2", thought_path_2)
builder.add_node("thought_path_3", thought_path_3)

builder.add_node("evaluator", evaluator)

# ============================================================
# STEP 10 — PARALLEL EXECUTION
# ============================================================

builder.add_edge(START, "thought_path_1")
builder.add_edge(START, "thought_path_2")
builder.add_edge(START, "thought_path_3")

builder.add_edge("thought_path_1", "evaluator")
builder.add_edge("thought_path_2", "evaluator")
builder.add_edge("thought_path_3", "evaluator")

builder.add_edge("evaluator", END)

# ============================================================
# STEP 11 — COMPILE GRAPH
# ============================================================

graph = builder.compile()

# ============================================================
# STEP 12 — VISUALIZE GRAPH
# ============================================================

display(
    Image(
        graph.get_graph().draw_mermaid_png()
    )
)

# ============================================================
# STEP 13 — RUN WORKFLOW
# ============================================================

result = graph.invoke(
    {
        "problem":
        """
        How should a retail company use AI
        to improve profitability?
        """
    }
)

# ============================================================
# STEP 14 — PRINT RESULTS
# ============================================================

print("\nFINAL DECISION\n")
print("=" * 60)
print(result["final_decision"])